# AI-Based Job & Skill Recommendation — Hybrid TF-IDF + SBERT Pipeline

**Architecture:**  
```
User Query
    │
    ▼
Stage 1 ── TF-IDF + Jaccard ──▶ top-K candidates  (fast, keyword-exact)
    │
    ▼
Stage 2 ── SBERT re-ranking  ──▶ top-N final results (semantic, cross-domain)
    │
    ▼
Hybrid score  =  α·tfidf  +  β·sbert  +  γ·jaccard
```

**Why hybrid?**
- TF-IDF is precise on exact skill keywords (`python`, `sql`) but blind to synonyms
- SBERT captures semantic equivalence (`Node.js` ≈ `server-side JavaScript`) but is slower
- Combining both outperforms either alone on Precision@5 / Recall@5 / nDCG@5

**References:**
- Ajjam & Al-Raweshidy (2026): TF-IDF cosine for job-matching
- Reimers & Gurevych (2019): Sentence-BERT (SBERT) semantic similarity
- Alsaif et al. (2022): Skill-weighted cosine similarity

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# 1. Install dependencies
# ============================================================
!pip -q install sentence-transformers scikit-learn pandas numpy

In [ ]:
# ============================================================
# 2. Imports
# ============================================================
import re
import time
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [ ]:
# ============================================================
# 3. Mount Drive & Load dataset
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/Machine Learning/DataSet/job_recommendation_dataset.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape: (50000, 7)
Columns: ['Job Title', 'Company', 'Location', 'Experience Level', 'Salary', 'Industry', 'Required Skills']


,Job Title,Company,Location,Experience Level,Salary,Industry,Required Skills
0,Early years teacher,Richardson Ltd,Sydney,Senior Level,87000.0,Healthcare,Pharmaceuticals
1,Counselling psychologist,"Ramos, Santiago and Stewart",San Francisco,Mid Level,50000.0,Marketing,"Google Ads, SEO, Content Writing"
2,Radio broadcast assistant,Franco Group,New York,Mid Level,77000.0,Healthcare,"Patient Care, Nursing, Medical Research, Pharm..."
3,"Designer, exhibition/display",Collins Inc,Berlin,Senior Level,90000.0,Software,Machine Learning
4,"Psychotherapist, dance movement",Barker Group,Sydney,Entry Level,112000.0,Healthcare,"Nursing, Medical Research, Pharmaceuticals"


In [ ]:
# ============================================================
# 4. Text Preprocessing
# ============================================================
SKILL_SYNONYMS = {
    'js': 'javascript', 'ts': 'typescript', 'py': 'python',
    'ml': 'machine learning', 'dl': 'deep learning',
    'nlp': 'natural language processing', 'cv': 'computer vision',
    'k8s': 'kubernetes', 'gcp': 'google cloud platform',
    'iac': 'infrastructure as code', 'cicd': 'continuous integration',
    'ui': 'user interface', 'ux': 'user experience',
    'bi': 'business intelligence', 'pm': 'project management',
}

def expand_synonyms(text: str) -> str:
    tokens = text.lower().split()
    return ' '.join(SKILL_SYNONYMS.get(t, t) for t in tokens)

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = expand_synonyms(text)
    text = re.sub(r"[^a-z0-9,+# ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

TEXT_COLS = ["Job Title", "Industry", "Experience Level", "Required Skills", "Location"]
for col in TEXT_COLS:
    df[col] = df[col].astype(str).fillna("")

# Weighted combined text — skills repeated 2x to boost their TF signal
# (mirrors kSkillWeight = 2.0 in the Flutter TF-IDF engine)
df["combined_text"] = (
    df["Job Title"] + " " +
    df["Industry"] + " " +
    df["Experience Level"] + " " +
    df["Required Skills"] + " " +   # first pass
    df["Required Skills"] + " " +   # second pass — skill weight boost
    df["Location"]
).apply(clean_text)

print("Sample combined_text:")
display(df[["Job Title", "Required Skills", "combined_text"]].head(3))

Sample combined_text:


,Job Title,Required Skills,combined_text
0,Early years teacher,Pharmaceuticals,early years teacher healthcare senior level ph...
1,Counselling psychologist,"Google Ads, SEO, Content Writing",counselling psychologist marketing mid level g...
2,Radio broadcast assistant,"Patient Care, Nursing, Medical Research, Pharm...",radio broadcast assistant healthcare mid level...


In [ ]:
# ============================================================
# 5. Stage 1 — TF-IDF Index (keyword-exact, fast)
# ============================================================
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=20_000,
    sublinear_tf=True,    # log(1+tf) — dampens high-frequency terms like BM25 k1
)
tfidf_matrix = tfidf_vectorizer.fit_transform(df["combined_text"])

print(f"TF-IDF matrix: {tfidf_matrix.shape}")

TF-IDF matrix: (50000, 3783)


In [ ]:
# ============================================================
# 6. Stage 2 — SBERT Index (semantic, slower, built once)
# ============================================================
# all-MiniLM-L6-v2: fast (384-dim), strong on skill/job similarity.
# For higher accuracy at the cost of speed use 'all-mpnet-base-v2' (768-dim).
SBERT_MODEL_NAME = "all-MiniLM-L6-v2"

print(f"Loading SBERT model: {SBERT_MODEL_NAME} ...")
sbert_model = SentenceTransformer(SBERT_MODEL_NAME)

print("Encoding job corpus with SBERT (runs once, ~30s for 10k rows) ...")
t0 = time.time()
sbert_job_embeddings = sbert_model.encode(
    df["combined_text"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # unit-length → dot product == cosine
)
print(f"SBERT index ready in {time.time()-t0:.1f}s  |  shape: {sbert_job_embeddings.shape}")

Loading SBERT model: all-MiniLM-L6-v2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding job corpus with SBERT (runs once, ~30s for 10k rows) ...


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

SBERT index ready in 750.9s  |  shape: (50000, 384)


In [ ]:
# ============================================================
# 7. Utility helpers
# ============================================================
def normalise_skills(skills_text: str) -> set:
    return {s.strip().lower() for s in str(skills_text).split(",") if s.strip()}

def build_user_query(
    target_role: str = "",
    skills: list = None,
    industry: str = "",
    level: str = "",
    location: str = "",
) -> str:
    skills = skills or []
    # Double the skills string to match the corpus skill-weight boost (§4)
    skill_str = ", ".join(skills)
    parts = [target_role, industry, level, location, skill_str, skill_str]
    return clean_text(" ".join(p for p in parts if p.strip()))

def jaccard(user_skills: set, job_skills_text: str) -> float:
    b = normalise_skills(job_skills_text)
    if not user_skills and not b:
        return 0.0
    return len(user_skills & b) / len(user_skills | b)

## Core Hybrid Recommendation Function

**Two-stage pipeline:**
1. TF-IDF retrieves `candidate_pool` (default 100) cheapest candidates
2. SBERT re-ranks those candidates with semantic similarity
3. Hybrid score blends all three signals with configurable weights `(α, β, γ)`

| Weight | Signal | Strength |
|--------|--------|----------|
| `alpha_tfidf` (0.40) | keyword overlap | exact skill terms |
| `beta_sbert` (0.45) | semantic similarity | paraphrases, related tech |
| `gamma_jaccard` (0.15) | set intersection | completeness check |

In [ ]:
# ============================================================
# 8. Hybrid recommendation engine
# ============================================================
OUTPUT_COLS = [
    "Job Title", "Company", "Location", "Experience Level",
    "Salary", "Industry", "Required Skills",
    "tfidf_score", "sbert_score", "jaccard_score", "hybrid_score",
]

def recommend_jobs_hybrid(
    user_query: str,
    user_skills: list,
    top_n: int = 10,
    candidate_pool: int = 100,   # Stage-1 TF-IDF pool size before SBERT re-rank
    alpha_tfidf: float = 0.40,   # TF-IDF weight
    beta_sbert: float = 0.45,    # SBERT semantic weight
    gamma_jaccard: float = 0.15, # Jaccard set-overlap weight
) -> pd.DataFrame:
    """
    Hybrid TF-IDF + SBERT job recommender.

    Stage 1 (TF-IDF): cheaply narrows the full corpus to `candidate_pool`
    best keyword-matching candidates. This avoids running SBERT's encoder
    over every job in the dataset.

    Stage 2 (SBERT re-rank): encodes only the user query (1 forward pass),
    then scores it against the *pre-computed* job embeddings for the pool.
    Because job embeddings are already unit-normalised, scoring is a fast
    numpy dot product.

    Final score = alpha_tfidf·tfidf + beta_sbert·sbert + gamma_jaccard·jaccard
    """
    assert abs(alpha_tfidf + beta_sbert + gamma_jaccard - 1.0) < 1e-6, (
        f"Weights must sum to 1.0, got {alpha_tfidf + beta_sbert + gamma_jaccard:.4f}"
    )

    user_skills_set = {s.strip().lower() for s in user_skills if s.strip()}
    query_clean = clean_text(user_query)

    # ── Stage 1: TF-IDF retrieval ──────────────────────────────────────────
    tfidf_vec = tfidf_vectorizer.transform([query_clean])
    tfidf_scores = cosine_similarity(tfidf_vec, tfidf_matrix).flatten()

    # Take top-candidate_pool indices (unsorted is fine; we sort next)
    pool_idx = np.argpartition(tfidf_scores, -candidate_pool)[-candidate_pool:]

    # ── Stage 2: SBERT re-ranking (only over the pool) ────────────────────
    user_emb = sbert_model.encode(
        [query_clean],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    # Dot product of two unit-normalised vectors == cosine similarity
    pool_sbert_scores = (user_emb @ sbert_job_embeddings[pool_idx].T).flatten()

    # ── Jaccard scores for the pool ────────────────────────────────────────
    pool_jaccard_scores = np.array([
        jaccard(user_skills_set, df.iloc[i]["Required Skills"])
        for i in pool_idx
    ])

    # ── Hybrid blend ──────────────────────────────────────────────────────
    pool_tfidf_scores = tfidf_scores[pool_idx]
    pool_hybrid_scores = (
        alpha_tfidf   * pool_tfidf_scores +
        beta_sbert    * pool_sbert_scores +
        gamma_jaccard * pool_jaccard_scores
    )

    # ── Assemble result DataFrame ─────────────────────────────────────────
    result = df.iloc[pool_idx].copy().reset_index(drop=True)
    result["tfidf_score"]   = pool_tfidf_scores
    result["sbert_score"]   = pool_sbert_scores
    result["jaccard_score"] = pool_jaccard_scores
    result["hybrid_score"]  = pool_hybrid_scores

    return (
        result
        .sort_values("hybrid_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
        [OUTPUT_COLS]
    )

print("✓ recommend_jobs_hybrid() ready")

✓ recommend_jobs_hybrid() ready


In [ ]:
# ============================================================
# 9. Baseline TF-IDF-only recommender (for comparison)
# ============================================================
def recommend_jobs_tfidf(
    user_query: str,
    user_skills: list,
    top_n: int = 10,
    alpha: float = 0.80,
    beta: float = 0.20,
) -> pd.DataFrame:
    user_skills_set = {s.strip().lower() for s in user_skills if s.strip()}
    query_clean = clean_text(user_query)

    tfidf_vec = tfidf_vectorizer.transform([query_clean])
    cos_scores = cosine_similarity(tfidf_vec, tfidf_matrix).flatten()
    jacc_scores = df["Required Skills"].apply(
        lambda x: jaccard(user_skills_set, x)
    ).values
    final_scores = alpha * cos_scores + beta * jacc_scores

    out = df.copy()
    out["tfidf_score"]   = cos_scores
    out["sbert_score"]   = 0.0
    out["jaccard_score"] = jacc_scores
    out["hybrid_score"]  = final_scores
    return (
        out.sort_values("hybrid_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
        [OUTPUT_COLS]
    )

print("✓ recommend_jobs_tfidf() ready (baseline)")

✓ recommend_jobs_tfidf() ready (baseline)


In [ ]:
# ============================================================
# 10. SBERT-only recommender (for comparison)
# ============================================================
def recommend_jobs_sbert(
    user_query: str,
    user_skills: list,
    top_n: int = 10,
    alpha: float = 0.85,
    beta: float = 0.15,
) -> pd.DataFrame:
    user_skills_set = {s.strip().lower() for s in user_skills if s.strip()}
    query_clean = clean_text(user_query)

    user_emb = sbert_model.encode(
        [query_clean],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    sem_scores = (user_emb @ sbert_job_embeddings.T).flatten()
    jacc_scores = df["Required Skills"].apply(
        lambda x: jaccard(user_skills_set, x)
    ).values
    final_scores = alpha * sem_scores + beta * jacc_scores

    out = df.copy()
    out["tfidf_score"]   = 0.0
    out["sbert_score"]   = sem_scores
    out["jaccard_score"] = jacc_scores
    out["hybrid_score"]  = final_scores
    return (
        out.sort_values("hybrid_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
        [OUTPUT_COLS]
    )

print("✓ recommend_jobs_sbert() ready (semantic baseline)")

✓ recommend_jobs_sbert() ready (semantic baseline)


## Skill-Gap Analysis

In [ ]:
# ============================================================
# 11. Skill-gap analysis (unchanged — works with any recommender output)
# ============================================================
def skill_gap_analysis(user_skills: list, recommended_df: pd.DataFrame) -> pd.DataFrame:
    user_set = {s.strip().lower() for s in user_skills if s.strip()}
    rows = []
    for _, row in recommended_df.iterrows():
        required = normalise_skills(row["Required Skills"])
        rows.append({
            "Job Title":          row["Job Title"],
            "Matched Skills":     ", ".join(sorted(required & user_set)),
            "Missing Skills":     ", ".join(sorted(required - user_set)),
            "Missing Skill Count": len(required - user_set),
            "hybrid_score":       round(row["hybrid_score"], 4),
        })
    return pd.DataFrame(rows)

## Example — Run all three models

In [ ]:
# ============================================================
# 12. Example query
# ============================================================
CANDIDATE_ROLE     = "data analyst"
CANDIDATE_SKILLS   = ["python", "sql", "excel"]
CANDIDATE_INDUSTRY = "software"
CANDIDATE_LEVEL    = "entry level"
CANDIDATE_LOCATION = "london"

user_query = build_user_query(
    target_role=CANDIDATE_ROLE,
    skills=CANDIDATE_SKILLS,
    industry=CANDIDATE_INDUSTRY,
    level=CANDIDATE_LEVEL,
    location=CANDIDATE_LOCATION,
)
print("User query string:", user_query)

# ── TF-IDF baseline ───────────────────────────────────────────────────────
print("\n=== TF-IDF only ===")
tfidf_results = recommend_jobs_tfidf(user_query, CANDIDATE_SKILLS, top_n=10)
display(tfidf_results)

# ── SBERT only ────────────────────────────────────────────────────────────
print("\n=== SBERT only ===")
sbert_results = recommend_jobs_sbert(user_query, CANDIDATE_SKILLS, top_n=10)
display(sbert_results)

# ── Hybrid (TF-IDF Stage-1 → SBERT re-rank) ──────────────────────────────
print("\n=== Hybrid TF-IDF + SBERT ===")
hybrid_results = recommend_jobs_hybrid(
    user_query, CANDIDATE_SKILLS,
    top_n=10,
    candidate_pool=100,
    alpha_tfidf=0.40,
    beta_sbert=0.45,
    gamma_jaccard=0.15,
)
display(hybrid_results)

User query string: data analyst software entry level london python, sql, excel python, sql, excel

=== TF-IDF only ===


,Job Title,Company,Location,Experience Level,Salary,Industry,Required Skills,tfidf_score,sbert_score,jaccard_score,hybrid_score
0,Cytogeneticist,Wise-Gutierrez,London,Entry Level,130000.0,Finance,"Python, SQL, Excel",0.563457,0.0,1.0,0.650765
1,Intelligence analyst,Flynn-Lee,New York,Mid Level,50000.0,Finance,"Python, SQL, Excel",0.541883,0.0,1.0,0.633506
2,Proofreader,"Riley, Allen and Buck",Berlin,Entry Level,115000.0,Finance,"Python, SQL, Excel",0.535966,0.0,1.0,0.628772
3,Retail manager,Glover-Henry,Sydney,Mid Level,86000.0,Finance,"Python, SQL, Excel",0.534649,0.0,1.0,0.627719
4,Lawyer,Jones-Moore,Berlin,Entry Level,79000.0,Finance,"Python, SQL, Excel",0.533644,0.0,1.0,0.626915
5,Energy engineer,Jennings-Flowers,Toronto,Entry Level,130000.0,Finance,"Python, SQL, Excel",0.532333,0.0,1.0,0.625866
6,Artist,Sullivan-Richardson,Bangalore,Entry Level,137000.0,Finance,"Excel, Python, SQL",0.528787,0.0,1.0,0.623030
7,Dietitian,Fields PLC,New York,Entry Level,70000.0,Finance,"Python, SQL, Excel",0.525858,0.0,1.0,0.620686
8,Financial adviser,Suarez-Williamson,Berlin,Entry Level,139000.0,Finance,"Python, SQL, Excel",0.525180,0.0,1.0,0.620144
9,Psychotherapist,Alvarez PLC,Toronto,Mid Level,101000.0,Finance,"SQL, Excel, Python",0.519425,0.0,1.0,0.615540



=== SBERT only ===


,Job Title,Company,Location,Experience Level,Salary,Industry,Required Skills,tfidf_score,sbert_score,jaccard_score,hybrid_score
0,"Engineer, production","Castaneda, Carpenter and Mckinney",London,Entry Level,56000.0,Finance,"Excel, SQL, Python",0.0,0.879195,1.0,0.897316
1,"Investment banker, corporate",Hill-Marquez,London,Entry Level,54000.0,Finance,"SQL, Excel, Python",0.0,0.875393,1.0,0.894084
2,"Engineer, control and instrumentation","Gonzalez, Wells and Williams",London,Mid Level,131000.0,Finance,"Excel, Python, SQL",0.0,0.856824,1.0,0.878300
3,Intelligence analyst,Flynn-Lee,New York,Mid Level,50000.0,Finance,"Python, SQL, Excel",0.0,0.848386,1.0,0.871128
4,Land,"Anderson, Spencer and Gray",Toronto,Entry Level,131000.0,Finance,"Python, Excel, SQL",0.0,0.837554,1.0,0.861921
5,"Secretary, company","Scott, Cortez and Leonard",Toronto,Entry Level,95000.0,Finance,"Excel, SQL, Python",0.0,0.837483,1.0,0.861861
6,Counsellor,King PLC,London,Senior Level,121000.0,Finance,"SQL, Python, Excel",0.0,0.837058,1.0,0.861499
7,"Engineer, automotive",Davis-Nguyen,New York,Entry Level,47000.0,Finance,"SQL, Excel, Python",0.0,0.837023,1.0,0.861469
8,Hydrologist,"Dudley, Wade and Hartman",London,Senior Level,78000.0,Finance,"Python, Excel, SQL",0.0,0.836617,1.0,0.861124
9,Corporate investment banker,Curtis-Austin,Bangalore,Entry Level,102000.0,Finance,"SQL, Excel, Python",0.0,0.835485,1.0,0.860162



=== Hybrid TF-IDF + SBERT ===


,Job Title,Company,Location,Experience Level,Salary,Industry,Required Skills,tfidf_score,sbert_score,jaccard_score,hybrid_score
0,Intelligence analyst,Flynn-Lee,New York,Mid Level,50000.0,Finance,"Python, SQL, Excel",0.541883,0.848386,1.0,0.748527
1,Financial adviser,Suarez-Williamson,Berlin,Entry Level,139000.0,Finance,"Python, SQL, Excel",0.525180,0.835171,1.0,0.735899
2,Lawyer,Jones-Moore,Berlin,Entry Level,79000.0,Finance,"Python, SQL, Excel",0.533644,0.826884,1.0,0.735555
3,Retail manager,Glover-Henry,Sydney,Mid Level,86000.0,Finance,"Python, SQL, Excel",0.534649,0.824870,1.0,0.735051
4,Energy engineer,Jennings-Flowers,Toronto,Entry Level,130000.0,Finance,"Python, SQL, Excel",0.532333,0.822026,1.0,0.732845
5,"Investment banker, corporate",Hill-Marquez,London,Entry Level,54000.0,Finance,"SQL, Excel, Python",0.471485,0.875393,1.0,0.732521
6,Chemical engineer,Cuevas Group,Berlin,Senior Level,110000.0,Finance,"Python, SQL, Excel",0.512250,0.817928,1.0,0.722968
7,Cytogeneticist,Wise-Gutierrez,London,Entry Level,130000.0,Finance,"Python, SQL, Excel",0.563457,0.771246,1.0,0.722443
8,Dietitian,Fields PLC,New York,Entry Level,70000.0,Finance,"Python, SQL, Excel",0.525858,0.804503,1.0,0.722370
9,Forensic scientist,"Oconnor, Hernandez and Shaw",London,Senior Level,48000.0,Finance,"SQL, Excel, Python",0.507743,0.819929,1.0,0.722065


In [ ]:
# ============================================================
# 13. Skill-gap analysis on hybrid results
# ============================================================
gap_df = skill_gap_analysis(CANDIDATE_SKILLS, hybrid_results)
display(gap_df)

,Job Title,Matched Skills,Missing Skills,Missing Skill Count,hybrid_score
0,Intelligence analyst,"excel, python, sql",,0,0.7485
1,Financial adviser,"excel, python, sql",,0,0.7359
2,Lawyer,"excel, python, sql",,0,0.7356
3,Retail manager,"excel, python, sql",,0,0.7351
4,Energy engineer,"excel, python, sql",,0,0.7328
5,"Investment banker, corporate","excel, python, sql",,0,0.7325
6,Chemical engineer,"excel, python, sql",,0,0.7230
7,Cytogeneticist,"excel, python, sql",,0,0.7224
8,Dietitian,"excel, python, sql",,0,0.7224
9,Forensic scientist,"excel, python, sql",,0,0.7221


## Evaluation — Compare TF-IDF vs SBERT vs Hybrid

Uses **proxy evaluation** (ground-truth approximation):
each test query is built from a sampled job row;
a result is *relevant* if its `Job Title` or `Industry` matches the source row.

In [ ]:
# ============================================================
# 14. Proxy evaluation harness
# ============================================================
def precision_at_k(recommended_titles: list, relevant_title: str, k: int) -> float:
    hits = sum(
        1 for t in recommended_titles[:k]
        if relevant_title.lower() in t.lower() or t.lower() in relevant_title.lower()
    )
    return hits / k

def recall_at_k(recommended_titles: list, relevant_title: str, k: int) -> float:
    hits = sum(
        1 for t in recommended_titles[:k]
        if relevant_title.lower() in t.lower() or t.lower() in relevant_title.lower()
    )
    return float(min(hits, 1))  # binary: did we find at least one?

def ndcg_at_k(recommended_titles: list, relevant_title: str, k: int) -> float:
    dcg, idcg = 0.0, 1.0  # ideal = relevant item at position 1
    for i, t in enumerate(recommended_titles[:k], start=1):
        if relevant_title.lower() in t.lower() or t.lower() in relevant_title.lower():
            dcg += 1.0 / np.log2(i + 1)
    return dcg / idcg if idcg > 0 else 0.0

def mrr(recommended_titles: list, relevant_title: str) -> float:
    for i, t in enumerate(recommended_titles, start=1):
        if relevant_title.lower() in t.lower() or t.lower() in relevant_title.lower():
            return 1.0 / i
    return 0.0

def run_evaluation(
    recommender_fn,
    n_queries: int = 50,
    k: int = 5,
    seed: int = 42,
    **kwargs,
) -> dict:
    """
    Proxy evaluation: sample `n_queries` jobs; query with their skills;
    check if the source job title appears in the top-k results.
    """
    rng = np.random.default_rng(seed)
    sample_idx = rng.choice(len(df), size=n_queries, replace=False)

    p_scores, r_scores, ndcg_scores, mrr_scores = [], [], [], []

    for idx in sample_idx:
        row = df.iloc[idx]
        skills = [s.strip() for s in str(row["Required Skills"]).split(",") if s.strip()]
        # Partially mask skills: use only first half to simulate a real user
        query_skills = skills[:max(1, len(skills)//2)]
        query = build_user_query(
            target_role=row["Job Title"],
            skills=query_skills,
            industry=row["Industry"],
            level=row["Experience Level"],
        )
        recs = recommender_fn(query, query_skills, top_n=k, **kwargs)
        titles = recs["Job Title"].tolist()
        ground_truth = row["Job Title"]

        p_scores.append(precision_at_k(titles, ground_truth, k))
        r_scores.append(recall_at_k(titles, ground_truth, k))
        ndcg_scores.append(ndcg_at_k(titles, ground_truth, k))
        mrr_scores.append(mrr(titles, ground_truth))

    return {
        f"Precision@{k}": round(np.mean(p_scores), 4),
        f"Recall@{k}":    round(np.mean(r_scores), 4),
        f"nDCG@{k}":      round(np.mean(ndcg_scores), 4),
        "MRR":            round(np.mean(mrr_scores), 4),
    }

print("✓ Evaluation harness ready")

✓ Evaluation harness ready


In [ ]:
# ============================================================
# 15. Run evaluation across all three models
# ============================================================
K = 5
N_QUERIES = 50

print(f"Evaluating {N_QUERIES} proxy queries at k={K} ...")

results_tfidf = run_evaluation(
    recommend_jobs_tfidf, n_queries=N_QUERIES, k=K
)
print("TF-IDF done")

results_sbert = run_evaluation(
    recommend_jobs_sbert, n_queries=N_QUERIES, k=K
)
print("SBERT done")

results_hybrid = run_evaluation(
    recommend_jobs_hybrid, n_queries=N_QUERIES, k=K,
    candidate_pool=100, alpha_tfidf=0.40, beta_sbert=0.45, gamma_jaccard=0.15,
)
print("Hybrid done")

# ── Summary table ─────────────────────────────────────────────────────────
eval_df = pd.DataFrame(
    [results_tfidf, results_sbert, results_hybrid],
    index=["TF-IDF", "SBERT", "Hybrid"]
)

print("\n===== Evaluation Summary =====")
display(eval_df.style.highlight_max(axis=0, color='#d4edda'))

Evaluating 50 proxy queries at k=5 ...
TF-IDF done
SBERT done
Hybrid done

===== Evaluation Summary =====


,Precision@5,Recall@5,nDCG@5,MRR
TF-IDF,0.572000,0.940000,1.843000,0.875000
SBERT,0.296000,0.720000,1.065800,0.654700
Hybrid,0.540000,0.960000,1.763500,0.843000


## Weight Ablation — Find the best α / β / γ

In [ ]:
# ============================================================
# 16. Ablation: sweep over alpha_tfidf / beta_sbert
#     (gamma_jaccard = 1 - alpha - beta)
# ============================================================
print("Running weight ablation (this takes ~2-3 min) ...")

ablation_rows = []
for alpha in np.arange(0.1, 0.8, 0.1):
    for beta in np.arange(0.1, 0.8 - alpha + 0.05, 0.1):
        gamma = round(1.0 - alpha - beta, 6)
        if gamma < 0.05:
            continue
        scores = run_evaluation(
            recommend_jobs_hybrid, n_queries=30, k=5,
            candidate_pool=80,
            alpha_tfidf=round(alpha, 2),
            beta_sbert=round(beta, 2),
            gamma_jaccard=round(gamma, 2),
            seed=99,
        )
        ablation_rows.append({
            "alpha_tfidf": round(alpha, 2),
            "beta_sbert": round(beta, 2),
            "gamma_jaccard": round(gamma, 2),
            **scores,
        })

ablation_df = pd.DataFrame(ablation_rows).sort_values("nDCG@5", ascending=False)
print("\nTop 10 weight combinations by nDCG@5:")
display(ablation_df.head(10))

Running weight ablation (this takes ~2-3 min) ...

Top 10 weight combinations by nDCG@5:


,alpha_tfidf,beta_sbert,gamma_jaccard,Precision@5,Recall@5,nDCG@5,MRR
27,0.7,0.1,0.2,0.6200,0.9000,1.9638,0.8361
26,0.6,0.2,0.2,0.5867,0.9000,1.8732,0.8317
24,0.5,0.3,0.2,0.5333,0.8667,1.7301,0.7983
21,0.4,0.4,0.2,0.4667,0.8333,1.5674,0.7900
17,0.3,0.5,0.2,0.4533,0.8333,1.5126,0.7844
25,0.6,0.1,0.3,0.4000,0.8333,1.4146,0.7800
12,0.2,0.6,0.2,0.4067,0.8000,1.4127,0.7778
6,0.1,0.7,0.2,0.3867,0.8000,1.3740,0.7778
23,0.5,0.2,0.3,0.3800,0.7667,1.3474,0.7500
20,0.4,0.3,0.3,0.3467,0.7667,1.2649,0.7417


## Export SBERT embeddings for the Flutter API

The Flask API (`sbert_api.py`) loads these pre-computed embeddings at startup,
so it never has to re-encode the entire job corpus on each request.

In [ ]:
# ============================================================
# 17. Save embeddings + job index for the Flask API
# ============================================================
import json

EXPORT_DIR = "/content/drive/MyDrive/ML Project/sbert_index"
import os; os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. SBERT embeddings matrix  (n_jobs × 384)
np.save(f"{EXPORT_DIR}/job_embeddings.npy", sbert_job_embeddings)

# 2. Lightweight job index for the API to return structured results
job_index = df[["Job Title", "Company", "Location",
                "Experience Level", "Salary", "Industry",
                "Required Skills", "combined_text"]].to_dict(orient="records")
with open(f"{EXPORT_DIR}/job_index.json", "w") as f:
    json.dump(job_index, f)

print(f"Saved {sbert_job_embeddings.shape[0]} embeddings to {EXPORT_DIR}/")
print("Files: job_embeddings.npy, job_index.json")

Saved 50000 embeddings to /content/drive/MyDrive/ML Project/sbert_index/
Files: job_embeddings.npy, job_index.json


### 18. Manual Testing
Use this cell to test the recommendation engine with custom inputs.